# Notebook 02: Person Re-ID Training
**Multi-Camera Tracking System — Kaggle Training Pipeline**

Train person re-identification models on Market-1501 dataset using torchreid.

**Models**: OSNet-x1.0 (512-dim) + ResNet50-IBN-a (2048-dim)  
**Runtime**: GPU T4/P100 | **Duration**: ~3-4 hours total

## Target Metrics
| Model | mAP | Rank-1 |
|-------|-----|--------|
| OSNet-x1.0 | ≥85% | ≥94% |
| ResNet50-IBN-a | ≥85% | ≥93% |

In [ ]:
!pip install -q torchreid gdown matplotlib pandas

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/reid_models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training configuration
# NOTE: root must be a WRITABLE directory — /kaggle/input is read-only
CONFIG = {
    "dataset": "market1501",
    "market1501_root": "/kaggle/working/data",
    "batch_size_train": 64,
    "batch_size_test": 128,
    "num_workers": 2,
    "pin_memory": True,
}

print(f"\nDevice: {DEVICE}")
print(f"Output: {OUTPUT_DIR}")

## 2. Dataset Registration

In [ ]:
# Prepare data directory — /kaggle/input/ is read-only, torchreid needs a writable root
# torchreid Market1501 expects: {root}/market1501/ containing bounding_box_train/, query/, bounding_box_test/

DATA_ROOT = Path("/kaggle/working/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/
# Try both possible mount locations
possible_roots = [
    Path("/kaggle/input/datasets/pengcw1/market-1501"),
    Path("/kaggle/input/market-1501"),
]

input_dir = None
for root in possible_roots:
    if root.exists():
        input_dir = root
        break

if input_dir is None:
    # Last resort: search all of /kaggle/input for bounding_box_train
    import subprocess
    result = subprocess.run(["find", "/kaggle/input", "-maxdepth", "5", "-type", "d", "-name", "bounding_box_train"],
                            capture_output=True, text=True, timeout=30)
    print(f"Search results:\n{result.stdout}")
    raise RuntimeError(
        f"Market-1501 dataset not found. Tried: {[str(p) for p in possible_roots]}. "
        "Make sure the dataset 'pengcw1/market-1501' is attached to this notebook."
    )

print(f"Dataset found at: {input_dir}")
print("Contents:")
for p in sorted(input_dir.iterdir()):
    kind = f"{len(list(p.iterdir()))} items" if p.is_dir() else "file"
    print(f"  {p.name}/ [{kind}]" if p.is_dir() else f"  {p.name} [{kind}]")

# Find where bounding_box_train lives (nested under Market-1501-v15.09.15/)
market_data = None
for p in input_dir.rglob("bounding_box_train"):
    if p.is_dir():
        market_data = p.parent
        break

if market_data is None:
    raise RuntimeError(f"Could not find 'bounding_box_train' inside {input_dir}")

print(f"\nMarket-1501 data at: {market_data}")

# Symlink to writable location: /kaggle/working/data/market1501 -> actual data
symlink = DATA_ROOT / "market1501"
if symlink.exists() or symlink.is_symlink():
    symlink.unlink()
symlink.symlink_to(market_data)

# Verify
for subdir in ["bounding_box_train", "bounding_box_test", "query"]:
    d = symlink / subdir
    if d.exists():
        n = len(list(d.iterdir()))
        print(f"  {subdir}: {n} files")
    else:
        print(f"  WARNING: {subdir} NOT FOUND!")

print(f"\nData root ready: {DATA_ROOT}")

In [ ]:
import torchreid

# Register dataset — torchreid has built-in Market-1501 support
# We just need to point it to the right directory
datamanager = torchreid.data.ImageDataManager(
    root=CONFIG["market1501_root"],
    sources=["market1501"],
    targets=["market1501"],
    height=256,
    width=128,
    batch_size_train=CONFIG["batch_size_train"],
    batch_size_test=CONFIG["batch_size_test"],
    transforms=["random_flip", "random_crop", "random_erasing"],
    workers=CONFIG["num_workers"],
)

print(f"Train: {datamanager.num_train_pids} IDs, {datamanager.num_train_cams} cameras")

## 3. Model A: ResNet50-IBN-a Training

ResNet50 with Instance-Batch Normalization produces 2048-dimensional embeddings.
Good baseline with strong generalization across domains.

**Config**: 120 epochs, Adam lr=3.5e-4, warmup 10 epochs, triplet + cross-entropy + label smoothing

In [ ]:
# Build ResNet50-IBN-a model
model_resnet = torchreid.models.build_model(
    name="resnet50_ibn_a",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_resnet = model_resnet.to(DEVICE)

# Use both GPUs if available (T4x2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_resnet = nn.DataParallel(model_resnet)

# Optimizer and scheduler
optimizer_resnet = torchreid.optim.build_optimizer(
    model_resnet,
    optim="adam",
    lr=3.5e-4,
    weight_decay=5e-4,
)

scheduler_resnet = torchreid.optim.build_lr_scheduler(
    optimizer_resnet,
    lr_scheduler="cosine",
    max_epoch=120,
)

print(f"Model parameters: {sum(p.numel() for p in model_resnet.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_resnet.parameters() if p.requires_grad):,}")

In [ ]:
# Build training engine
engine_resnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model_resnet,
    optimizer=optimizer_resnet,
    scheduler=scheduler_resnet,
    label_smooth=True,
)

# Train
print("=" * 60)
print("Training ResNet50-IBN-a on Market-1501")
print("=" * 60)

start_time = time.time()
engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "resnet50_ibn_a"),
    max_epoch=120,
    eval_freq=20,
    print_freq=50,
    test_only=False,
    fixbase_epoch=10,  # warmup: freeze base for first 10 epochs
)
resnet_train_time = time.time() - start_time
print(f"\nTraining completed in {resnet_train_time / 3600:.1f} hours")

## 4. Evaluate ResNet50-IBN-a

In [ ]:
# Load best model and evaluate
# Note: engine.run(test_only=True) prints metrics but returns None
engine_resnet.run(
    save_dir=str(OUTPUT_DIR / "resnet50_ibn_a"),
    max_epoch=120,
    test_only=True,
)

print("\nResNet50-IBN-a evaluation complete. See mAP and Rank-1 metrics above.")

## 5. Model B: OSNet-x1.0 Training

OSNet (Omni-Scale Network) produces compact 512-dimensional embeddings.
Designed specifically for ReID with multi-scale feature aggregation.

**Config**: 150 epochs, Adam lr=3.5e-4, warmup 10 epochs, triplet + cross-entropy + label smoothing

In [ ]:
# Build OSNet-x1.0 model
model_osnet = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=datamanager.num_train_pids,
    loss="softmax",
    pretrained=True,
)
model_osnet = model_osnet.to(DEVICE)

# Use both GPUs if available (T4x2)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
    model_osnet = nn.DataParallel(model_osnet)

optimizer_osnet = torchreid.optim.build_optimizer(
    model_osnet,
    optim="adam",
    lr=3.5e-4,
    weight_decay=5e-4,
)

scheduler_osnet = torchreid.optim.build_lr_scheduler(
    optimizer_osnet,
    lr_scheduler="cosine",
    max_epoch=150,
)

print(f"Model parameters: {sum(p.numel() for p in model_osnet.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_osnet.parameters() if p.requires_grad):,}")

In [ ]:
engine_osnet = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model_osnet,
    optimizer=optimizer_osnet,
    scheduler=scheduler_osnet,
    label_smooth=True,
)

print("=" * 60)
print("Training OSNet-x1.0 on Market-1501")
print("=" * 60)

start_time = time.time()
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "osnet_x1_0"),
    max_epoch=150,
    eval_freq=20,
    print_freq=50,
    test_only=False,
    fixbase_epoch=10,
)
osnet_train_time = time.time() - start_time
print(f"\nTraining completed in {osnet_train_time / 3600:.1f} hours")

## 6. Evaluate OSNet-x1.0

In [ ]:
# Note: engine.run(test_only=True) prints metrics but returns None
engine_osnet.run(
    save_dir=str(OUTPUT_DIR / "osnet_x1_0"),
    max_epoch=150,
    test_only=True,
)

print("\nOSNet-x1.0 evaluation complete. See mAP and Rank-1 metrics above.")

## 7. Results Comparison

In [ ]:
# Results comparison
results_summary = {
    "resnet50_ibn_a": {
        "embedding_dim": 2048,
        "train_time_hours": resnet_train_time / 3600,
        "epochs": 120,
    },
    "osnet_x1_0": {
        "embedding_dim": 512,
        "train_time_hours": osnet_train_time / 3600,
        "epochs": 150,
    },
}

# Save results
results_path = OUTPUT_DIR / "person_reid_results.json"
with open(results_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print("=" * 60)
print("Person ReID Training Summary")
print("=" * 60)
print(f"{'Model':<20} {'Dim':>5} {'Epochs':>7} {'Time (h)':>9}")
print("-" * 60)
for name, info in results_summary.items():
    print(f"{name:<20} {info['embedding_dim']:>5} {info['epochs']:>7} {info['train_time_hours']:>9.1f}")

print(f"\nResults saved to: {results_path}")

## 8. Export Models

In [ ]:
# Copy best checkpoints to a clean export directory
export_dir = Path("/kaggle/working/exported_models")
export_dir.mkdir(parents=True, exist_ok=True)

models_to_export = {
    "person_resnet50_ibn_a_market1501.pth": OUTPUT_DIR / "resnet50_ibn_a" / "model" / "model-best.pth.tar",
    "person_osnet_x1_0_market1501.pth": OUTPUT_DIR / "osnet_x1_0" / "model" / "model-best.pth.tar",
}

for export_name, src_path in models_to_export.items():
    if src_path.exists():
        dst_path = export_dir / export_name
        shutil.copy2(src_path, dst_path)
        size_mb = dst_path.stat().st_size / (1024 * 1024)
        print(f"Exported: {export_name} ({size_mb:.1f} MB)")
    else:
        # Try alternative checkpoint naming
        alt_path = src_path.parent / "model.pth.tar-120"
        if alt_path.exists():
            shutil.copy2(alt_path, export_dir / export_name)
            print(f"Exported (alt): {export_name}")
        else:
            print(f"Warning: {src_path} not found")

# Save model metadata
metadata = {
    "task": "person_reid",
    "dataset": "market1501",
    "models": {
        "resnet50_ibn_a": {
            "architecture": "resnet50_ibn_a",
            "embedding_dim": 2048,
            "input_size": [256, 128],
            "file": "person_resnet50_ibn_a_market1501.pth",
        },
        "osnet_x1_0": {
            "architecture": "osnet_x1_0",
            "embedding_dim": 512,
            "input_size": [256, 128],
            "file": "person_osnet_x1_0_market1501.pth",
        },
    },
    "trained_at": datetime.now().isoformat(),
}

with open(export_dir / "person_reid_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nAll exports saved to: {export_dir}")
print("Download these files and place in your local project's models/reid/ directory.")

## Next Steps
1. **Download** the exported model files from `/kaggle/working/exported_models/`
2. Place them in your local project: `models/reid/`
3. Update `configs/default.yaml` with the model paths
4. Proceed to **Notebook 03** for vehicle ReID training

### Local Usage
```python
# In your local project:
import torchreid

model = torchreid.models.build_model(
    name="osnet_x1_0",
    num_classes=751,  # Market-1501 train IDs
    loss="softmax",
    pretrained=False,
)
torchreid.utils.load_pretrained_weights(model, "models/reid/person_osnet_x1_0_market1501.pth")
model.eval()
```